<center>
<img src="../../img/ods_stickers.jpg">
## [mlcourse.ai](https://mlcourse.ai) - دورة التعلم الآلي المفتوحة
    
المؤلفون: [إيليا باريشنيكوف](https://www.linkedin.com/in/baryshnikov-ilya/)، [مكسيم أوفاروف](https://www.linkedin.com/in/maxis42/)، و[يوري كاشنيتسكي](https://www.linkedin.com/in/festline/). تمت الترجمة والتحرير بواسطة [إنجا كايدانوفا](https://www.linkedin.com/in/inga-kaidanova-a92398b1/)، و[إيجور بولوسماك](https://www.linkedin.com/in/egor-polusmak/)، و[أناستازيا مانوخينا](https://www.linkedin.com/in/anastasiamanokhina/)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). يتم توزيع كل المحتوى بموجب ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).



# <center>المهمة رقم 2 (تجريبي). الحل
## <center>تحليل بيانات أمراض القلب والأوعية الدموية 
    
    
** نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a2-demo-analyzing-cardiovascular-data) + [الحل](https://www.kaggle.com/kashnitsky/a2-demo-analyzing-cardiovascular-data-solution).**



في هذا التدريب، سوف تجيب على الأسئلة المتعلقة بمجموعة البيانات المتعلقة بأمراض القلب والأوعية الدموية. لا تحتاج إلى تنزيل البيانات: فهي موجودة بالفعل في المستودع. هناك بعض المهام التي ستتطلب منك كتابة التعليمات البرمجية. أكملها ثم أجب عن الأسئلة في [النموذج](https://docs.google.com/forms/d/13cE_tSIb6hsScQvvWUJeu1MEHE5L6vnxQUbDYpXsf24).
#### مشكلة
التنبؤ بوجود أو عدم وجود أمراض القلب والأوعية الدموية (CVD) باستخدام نتائج فحص المريض.
#### وصف البيانات
هناك 3 أنواع من ميزات الإدخال:
- *الهدف*: معلومات واقعية؛
- *الفحص*: نتائج الفحص الطبي؛
- *ذاتي*: المعلومات المقدمة من قبل المريض.| ميزة | نوع متغير | متغير | نوع القيمة |
|---------|---------------------|------------|------------|
| العمر | ميزة الهدف | العمر | كثافة العمليات (أيام) |
| الارتفاع | ميزة الهدف | الارتفاع | كثافة العمليات (سم) |
| الوزن | ميزة الهدف | الوزن | تعويم (كجم) |
| الجنس | ميزة الهدف | الجنس | الكود الفئوي |
| ضغط الدم الانقباضي | ميزة الفحص | ap_hi | كثافة العمليات |
| ضغط الدم الانبساطي | ميزة الفحص | ap_lo | كثافة العمليات |
| الكولسترول | ميزة الفحص | الكولسترول | 1: عادي، 2: أعلى من الطبيعي، 3: أعلى بكثير من الطبيعي |
| الجلوكوز | ميزة الفحص | جلوك | 1: عادي، 2: أعلى من الطبيعي، 3: أعلى بكثير من الطبيعي |
| التدخين | ميزة ذاتية | دخان | ثنائي |
| تناول الكحول | ميزة ذاتية | الكو | ثنائي |
| النشاط البدني | ميزة ذاتية | نشط | ثنائي |
| وجود أو عدم وجود أمراض القلب والأوعية الدموية | المتغير المستهدف | القلب | ثنائي |
تم جمع جميع قيم مجموعة البيانات في وقت الفحص الطبي.



دعونا نتعرف على بياناتنا من خلال إجراء تحليل أولي للبيانات.
# الجزء الأول. تحليل البيانات الأولية
أولاً، سنقوم بتهيئة البيئة:


In [ ]:
# Import all required modules
# Disable warnings
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Import plotting modules
import seaborn as sns

sns.set()
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker

%config InlineBackend.figure_format = 'retina'


يجب عليك استخدام مكتبة `seaborn` للتحليل المرئي، لذلك دعونا نقوم بإعدادها أيضًا:


In [ ]:
# Tune the visual settings for figures in `seaborn`
sns.set_context(
    "notebook", font_scale=1.5, rc={"figure.figsize": (11, 8), "axes.titlesize": 18}
)

from matplotlib import rcParams

rcParams["figure.figsize"] = 11, 8


لتبسيط الأمر، سنعمل فقط مع الجزء التدريبي من مجموعة البيانات:


In [ ]:
df = pd.read_csv("../../data/mlbootcamp5_train.csv", sep=";")
print("Dataset size: ", df.shape)
df.head()


سيكون من المفيد إلقاء نظرة خاطفة على قيم المتغيرات لدينا.
 
لنقم بتحويل البيانات إلى تنسيق *طويل* ونوضح عدد قيم الميزات الفئوية باستخدام [`factorplot()`](https://seaborn.pydata.org/generated/seaborn.factorplot.html).


In [ ]:
df_uniques = pd.melt(
    frame=df,
    value_vars=["gender", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"],
)
df_uniques = (
    pd.DataFrame(df_uniques.groupby(["variable", "value"])["value"].count())
    .sort_index(level=[0, 1])
    .rename(columns={"value": "count"})
    .reset_index()
)

sns.factorplot(
    x="variable", y="count", hue="value", data=df_uniques, kind="bar", size=12
);


يمكننا أن نرى أن الفئات المستهدفة متوازنة. هذا عظيم!
دعونا نقسم مجموعة البيانات حسب القيم المستهدفة: في بعض الأحيان يمكنك على الفور تحديد الميزة الأكثر أهمية في المخطط.


In [ ]:
df_uniques = pd.melt(
    frame=df,
    value_vars=["gender", "cholesterol", "gluc", "smoke", "alco", "active"],
    id_vars=["cardio"],
)
df_uniques = (
    pd.DataFrame(df_uniques.groupby(["variable", "value", "cardio"])["value"].count())
    .sort_index(level=[0, 1])
    .rename(columns={"value": "count"})
    .reset_index()
)

sns.factorplot(
    x="variable",
    y="count",
    hue="value",
    col="cardio",
    data=df_uniques,
    kind="bar",
    size=9,
);

يمكنك أن ترى أن المتغير المستهدف يؤثر بشكل كبير على توزيع مستويات الكوليسترول والجلوكوز. هل هذه صدفة؟
الآن، لنحسب بعض الإحصائيات الخاصة بالقيم الفريدة للميزة:


In [ ]:
for c in df.columns:
    n = df[c].nunique()
    print(c)
    if n <= 3:
        print(n, sorted(df[c].value_counts().to_dict().items()))
    else:
        print(n)
    print(10 * "-")


وفي النهاية لدينا:
- 5 ميزات رقمية (باستثناء *المعرف*)؛
- 7 ميزات الفئوية.
- 70000 سجل في المجموع.



##1.1. الملاحظات الأساسية



** السؤال 1.1. (نقطة واحدة). كم عدد الرجال والنساء الموجودين في مجموعة البيانات هذه؟ لم يتم توضيح قيم ميزة `gender` (سواء كان الرقم "1" يشير إلى النساء أو الرجال) - اكتشف ذلك من خلال النظر في تحليل الارتفاع، بافتراض معقول أن الرجال أطول في المتوسط.**
1. 45530 امرأة و 24470 رجلاً
2. 45530 رجلاً و 24470 امرأة
3. 45470 امرأة و 24530 رجلاً
4. 45470 رجلاً و 24530 امرأة
**الجواب:** 1.



### الحل:



لنحسب متوسط الارتفاع لكلا القيمتين `gender`:


In [ ]:
df.groupby("gender")["height"].mean()


161 سم وحوالي 170 سم في المتوسط، لذلك نستنتج أن الجنس=1 يمثل الإناث، والجنس=2 – الذكور. وبالتالي فإن العينة تحتوي على 45530 امرأة و 24470 رجلاً.



** السؤال 1.2. (نقطة واحدة). من الذي يُبلغ في كثير من الأحيان عن تعاطي الكحول - الرجال أم النساء؟**
1. النساء
2. الرجال
**الجواب:** 2.



### الحل:


In [ ]:
df.groupby("gender")["alco"].mean()


حسنا...واضح :)



** السؤال 1.3. (نقطة واحدة). ما هو الفارق التقريبي بين نسب المدخنين بين الرجال والنساء؟**
1. 4
2. 16
3. 20
4. 24
**الجواب:** 3.



### الحل:


In [ ]:
df.groupby("gender")["smoke"].mean()

In [ ]:
round(
    100
    * (
        df.loc[df["gender"] == 2, "smoke"].mean()
        - df.loc[df["gender"] == 1, "smoke"].mean()
    )
)


** السؤال 1.4. (نقطة واحدة). ما هو الفرق المقرب بين متوسط ​​قيم العمر (بالشهور) لغير المدخنين والمدخنين؟ ستحتاج إلى معرفة وحدات الميزة `age` في مجموعة البيانات هذه.**
1. 5
2. 10
3. 15
4. 20
**الجواب:** 4.



### الحل:



يتم تحديد العمر هنا بالأيام.

In [ ]:
df.groupby("smoke")["age"].median() / 365.25


متوسط عمر المدخنين هو 52.4 سنة، ولغير المدخنين 54 سنة. ونرى أن الإجابة الصحيحة هي 20 شهرا. ولكن هنا طريقة لحساب ذلك بالضبط:


In [ ]:
(
    df[df["smoke"] == 0]["age"].median() - df[df["smoke"] == 1]["age"].median()
) / 365.25 * 12


##1.2. خرائط المخاطر
### المهمة:



على موقع الويب الخاص بالجمعية الأوروبية لأمراض القلب، يتم توفير [مقياس SCORE](https://www.escardio.org/Education/Practice-Tools/CVD-prevention-toolbox/SCORE-Risk-Charts). يتم استخدامه لحساب خطر الوفاة بسبب أمراض القلب والأوعية الدموية في السنوات العشر القادمة. ومن هنا:
<img src='../../img/SCORE_CVD_eng.png' width=60%>
دعونا نلقي نظرة على المستطيل العلوي الأيمن الذي يظهر مجموعة فرعية من الرجال المدخنين الذين تتراوح أعمارهم بين 60 إلى 65 عاما. (هذا ليس واضحا، ولكن القيم في الشكل تمثل الحد العلوي).
نرى القيمة 9 في الزاوية السفلية اليسرى من المستطيل و47 في الزاوية العلوية اليمنى. وهذا يعني أنه بالنسبة للأشخاص في هذه الفئة العمرية من الجنسين الذين يقل ضغطهم الانقباضي عن 120، يُقدر أن خطر الإصابة بأمراض القلب والأوعية الدموية أقل بخمس مرات من أولئك الذين يعانون من الضغط في هذه الفترة [160،180).
دعونا نحسب نفس النسبة، ولكن مع بياناتنا.
توضيحات:
- حساب ميزة ``age_years`` - العمر المقرب بالسنوات. لهذه المهمة، حدد الأشخاص الذين تتراوح أعمارهم بين 60 إلى 64 عامًا.
- تختلف فئات مستوى الكوليسترول في الشكل وفي بياناتنا. في الشكل، قيم ميزة ``cholesterol`` هي كما يلي: 4 مليمول/لتر $\rightarrow$ 1، 5-7 مليمول/لتر $\rightarrow$ 2، 8 مليمول/لتر $\rightarrow$ 3.
**السؤال 1.5. (2 نقطة). قم بحساب أجزاء المرضى (المصابين بأمراض القلب والأوعية الدموية) في جزأين موصوفين أعلاه. ما حاصل هذين الكسرين؟**
1. 1
2. 2
3. 3
4. 4
**الجواب:** 3.



### الحل:


In [ ]:
df["age_years"] = (df["age"] / 365.25).round().astype("int")

In [ ]:
df["age_years"].max()


أكبر الأشخاص سناً في العينة يبلغون من العمر 65 عاماً. هل هي صدفة؟ لا أعتقد ذلك! دعونا نختار الرجال المدخنين في سن [60،64].


In [ ]:
smoking_old_men = df[
    (df["gender"] == 2)
    & (df["age_years"] >= 60)
    & (df["age_years"] < 65)
    & (df["smoke"] == 1)
]

فإذا كان مستوى الكولسترول في هذه الفئة العمرية 1، والضغط الانقباضي أقل من 120، فإن نسبة المصابين بأمراض القلب والأوعية الدموية تكون 26%.


In [ ]:
smoking_old_men[
    (smoking_old_men["cholesterol"] == 1) & (smoking_old_men["ap_hi"] < 120)
]["cardio"].mean()


ومع ذلك، إذا كان مستوى الكوليسترول في هذه الفئة العمرية 3 سنوات، والضغط الانقباضي من 160 إلى 180، فإن نسبة الأشخاص المصابين بأمراض القلب والأوعية الدموية تبلغ 86٪.


In [ ]:
smoking_old_men[
    (smoking_old_men["cholesterol"] == 3)
    & (smoking_old_men["ap_hi"] >= 160)
    & (smoking_old_men["ap_hi"] < 180)
]["cardio"].mean()


ونتيجة لذلك، فإن الفرق هو حوالي 3 أضعاف. ليس 5 أضعاف، كما يخبرنا مقياس SCORE، ولكن من الممكن أن يعتمد خطر SCORE للإصابة بأمراض القلب والأوعية الدموية بشكل غير خطي على نسبة الأشخاص المرضى في الفئة العمرية المحددة.



##1.3. تحليل مؤشر كتلة الجسم
### المهمة:



إنشاء ميزة جديدة – مؤشر كتلة الجسم ([مؤشر كتلة الجسم](https://en.wikipedia.org/wiki/Body_mass_index)). للقيام بذلك، قسمة الوزن بالكيلوغرام على مربع الطول بالأمتار. يقال إن قيم مؤشر كتلة الجسم الطبيعية تتراوح من 18.5 إلى 25. 
**السؤال 1.6. (2 نقطة). اختر العبارات الصحيحة:.**
1. متوسط مؤشر كتلة الجسم في العينة يقع ضمن حدود القيم الطبيعية.
2. مؤشر كتلة الجسم للنساء أعلى في المتوسط ​​من الرجال.
3. الأشخاص الأصحاء لديهم، في المتوسط، مؤشر كتلة الجسم أعلى من الأشخاص المرضى.
4. في شريحة الرجال الأصحاء والذين لا يشربون، يكون مؤشر كتلة الجسم أقرب إلى القاعدة منه في شريحة النساء الأصحاء والذين لا يشربون الخمر
**الجواب:** 2 و 4
### الحل:


In [ ]:
df["BMI"] = df["weight"] / (df["height"] / 100) ** 2

In [ ]:
df["BMI"].median()


البيان الأول غير صحيح لأن متوسط مؤشر كتلة الجسم يتجاوز 25 نقطة.


In [ ]:
df.groupby("gender")["BMI"].median()


بيان الثواني صحيح – مؤشر كتلة الجسم للنساء أعلى في المتوسط.



العبارة الثالثة غير صحيحة.


In [ ]:
df.groupby(["gender", "alco", "cardio"])["BMI"].median().to_frame()


بمقارنة قيم مؤشر كتلة الجسم في الصفوف حيث `alco=0` و`cardio=0`، نرى أن العبارة الأخيرة صحيحة.



##1.4. بيانات التنظيف



### المهمة:
يمكننا أن نلاحظ أن البيانات ليست مثالية. أنه يحتوي على الكثير من "الأوساخ" وعدم الدقة. سنرى ذلك بشكل أفضل عندما نقوم بتصور البيانات.تصفية شرائح المرضى التالية (التي نعتبرها تحتوي على بيانات خاطئة)
- الضغط الانبساطي أعلى من الضغط الانقباضي. 
- الارتفاع أقل من 2.5% من النسبة المئوية (استخدم `pd.Series.quantile`. إذا لم تكن على دراية به - يرجى قراءة المستندات)
- أن يكون الطول أكثر من 97.5% بشكل صارم
- الوزن أقل من 2.5% بشكل صارم
- الوزن أكثر من 97.5% بشكل صارم
هذا ليس كل ما يمكننا القيام به لتنظيف البيانات، ولكن دعونا نتوقف هنا الآن.
**السؤال 1.7. (2 نقطة). ما هو عدد النسب المئوية للبيانات (المقربة) التي تخلصنا منها؟**
1. 8
2. 9
3. 10
4. 11
**الجواب:** 3
### الحل:


In [ ]:
df_to_remove = df[
    (df["ap_lo"] > df["ap_hi"])
    | (df["height"] < df["height"].quantile(0.025))
    | (df["height"] > df["height"].quantile(0.975))
    | (df["weight"] < df["weight"].quantile(0.025))
    | (df["weight"] > df["weight"].quantile(0.975))
]
print(df_to_remove.shape[0] / df.shape[0])

filtered_df = df[~df.index.isin(df_to_remove)]


لقد طرحنا حوالي 10% من البيانات الأصلية.



# الجزء 2. تحليل البيانات المرئية
##2.1. تصور مصفوفة الارتباط
لفهم الميزات بشكل أفضل، يمكنك إنشاء مصفوفة لمعاملات الارتباط بين الميزات. استخدم مجموعة البيانات الأولية (غير المصفاة).
### المهمة:
ارسم مصفوفة ارتباط باستخدام [`heatmap()`](http://seaborn.pydata.org/generated/seaborn.heatmap.html). يمكنك إنشاء المصفوفة باستخدام أدوات `pandas` القياسية مع المعلمات الافتراضية.
### الحل:


In [ ]:
# Calculate the correlation matrix
df = filtered_df.copy()

corr = df.corr(method="pearson")

# Create a mask to hide the upper triangle of the correlation matrix (which is symmetric)
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

f, ax = plt.subplots(figsize=(12, 9))

sns.heatmap(
    corr,
    mask=mask,
    vmax=1,
    center=0,
    annot=True,
    fmt=".1f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.5},
);


** السؤال 2.1. (نقطة واحدة).** أي زوج من الميزات يتمتع بأقوى ارتباط بيرسون بميزة *الجنس*؟
1. أمراض القلب والكوليسترول
2. الارتفاع، الدخان
3. دخان، ألكو
4. الطول والوزن



**الجواب:** 2.



##2.2. توزيع الطول للرجال والنساء
من خلال استكشافنا للقيم الفريدة، نعلم أن الجنس يتم ترميزه بواسطة القيمتين *1* و*2*. وعلى الرغم من عدم وجود تحديد لكيفية توافق هذه القيم مع الرجال والنساء، يمكنك معرفة ذلك بيانيًا من خلال النظر إلى متوسط ​​قيم الطول والوزن لكل قيمة في ميزة *الجنس*.
### مهمة:قم بإنشاء مخطط كمان للطول والجنس باستخدام [`violinplot()`](https://seaborn.pydata.org/generated/seaborn.violinplot.html). استخدم المعلمات:
- `hue` للتقسيم حسب الجنس؛
- `scale` لتقييم عدد السجلات لكل جنس.
لكي يتم عرض المخطط بشكل صحيح، تحتاج إلى تحويل `DataFrame` إلى تنسيق *طويل* باستخدام الدالة `melt()` من `pandas`. إليك [مثال آخر](https://stackoverflow.com/a/41575149/3338479) على ذلك.
### الحل:


In [ ]:
df_melt = pd.melt(frame=df, value_vars=["height"], id_vars=["gender"])

plt.figure(figsize=(12, 10))
ax = sns.violinplot(
    x="variable",
    y="value",
    hue="gender",
    palette="muted",
    split=True,
    data=df_melt,
    scale="count",
    scale_hue=False,
)


### المهمة:
قم بإنشاء اثنين من [`kdeplot`](https://seaborn.pydata.org/generated/seaborn.kdeplot.html) من ميزة *الارتفاع* لكل جنس على نفس المخطط. سترى الفرق بين الجنسين بشكل أوضح، لكن لن تتمكن من تقييم عدد السجلات في كل منهما.
### الحل:


In [ ]:
sns.FacetGrid(df, hue="gender", size=12).map(sns.kdeplot, "height").add_legend();


##2.3. ارتباط الرتبة
في معظم الحالات، يكون *معامل بيرسون للارتباط الخطي* أكثر من كافٍ لاكتشاف الأنماط في البيانات. 
ولكن دعنا نذهب أبعد من ذلك قليلاً ونحسب [ارتباط الرتبة](https://en.wikipedia.org/wiki/Rank_correlation). سيساعدنا ذلك في تحديد أزواج الميزات التي يكون فيها الترتيب الأدنى في السلسلة المتغيرة لميزة ما يسبق دائمًا الترتيب الأعلى في الميزة الأخرى (ولدينا العكس في حالة الارتباط السلبي).
### المهمة:
قم بحساب ورسم مصفوفة الارتباط باستخدام [معامل ارتباط رتبة سبيرمان](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient).
### الحل:


In [ ]:
# Calculate the correlation matrix
corr = df[
    ["id", "age", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc"]
].corr(method="spearman")

# Create a mask to hide the upper triangle of the correlation matrix (which is symmetric)
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

f, ax = plt.subplots(figsize=(12, 10))

# Plot the heatmap using the mask and correct aspect ratio
sns.heatmap(
    corr,
    mask=mask,
    vmax=1,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.5},
);


** السؤال 2.2. (نقطة واحدة).** أي زوج من الميزات لديه أقوى ارتباط سبيرمان؟
1. الطول والوزن
2. العمر والوزن
3. الكوليسترول، الجلوك
4. أمراض القلب والكولسترول
5. أب_هي، أب_لو
6. سموك، ألكو



**الجواب:** 5.



**السؤال 2.3. (نقطة واحدة).** لماذا ترتبط هذه الميزات ارتباطًا قويًا بالرتبة؟
1. عدم الدقة في البيانات (أخطاء الحصول على البيانات).
2. العلاقة خاطئة، ولا ينبغي أن تكون هذه الميزات مرتبطة ببعضها البعض.
3. طبيعة البيانات.


**الجواب:** 3.



##2.4. العمر
في السابق قمنا بحساب عمر المجيبين بالسنوات وقت الفحص.



### المهمة:
قم بإنشاء *مخطط العد* باستخدام [`countplot()`](http://seaborn.pydata.org/generated/seaborn.countplot.html)، مع تحديد العمر على المحور *X* وعدد الأشخاص على المحور *Y*. يجب أن تحتوي كل قيمة للعمر على عمودين يتوافقان مع أعداد الأشخاص في هذا العمر لكل فئة *كارديو*.
### الحل:


In [ ]:
sns.countplot(x="age_years", hue="cardio", data=df);


**السؤال 2.4. (نقطة واحدة).** في أي عمر يفوق عدد الأشخاص المصابين بأمراض القلب والأوعية الدموية عدد الأشخاص الذين لا يعانون من أمراض القلب والأوعية الدموية لأول مرة؟
1. 44
2. 55
3. 64
4. 70



**الجواب:** 2.